In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from torchvision import datasets
from torch.utils.data import Dataset, DataLoader
from torch.optim import SGD # Sử dụng lại SGD thay vì Adam

# Thiết lập cấu hình phần cứng tăng tốc
device = "cuda" if torch.cuda.is_available() else "cpu"

# Tải dữ liệu FMNIST cho cả tập Train và Validation
data_folder = '~/data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
tr_images = fmnist.data
tr_targets = fmnist.targets

val_fmnist = datasets.FashionMNIST(data_folder, download=True, train=False)
val_images = val_fmnist.data
val_targets = val_fmnist.targets

# Lớp Dataset tùy biến tích hợp chuẩn hóa /255
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float() / 255.0
        x = x.view(-1, 28 * 28)
        self.x, self.y = x, y
        
    def __getitem__(self, ix):
        x, y = self.x[ix], self.y[ix]
        return x.to(device), y.to(device)
        
    def __len__(self):
        return len(self.x)

def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    
    val = FMNISTDataset(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl

ModuleNotFoundError: No module named 'torch'

In [2]:
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

# CHÚ Ý: Vòng lặp tăng lên 10 kỷ nguyên (range(10)) theo tài liệu mới
for epoch in range(10):
    print(f"Epoch đang chạy (Dùng SGD): {epoch}")
    train_epoch_losses, train_epoch_accuracies = [], []
    
    # 1. Lan truyền dữ liệu tập Huấn luyện
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
    train_epoch_loss = np.array(train_epoch_losses).mean()
    
    # 2. Đánh giá Accuracy trên tập Huấn luyện
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    train_epoch_accuracy = np.mean(train_epoch_accuracies)
    
    # 3. Đánh giá tính toán độ lỗi và chính xác trên tập Xác thực
    for ix, batch in enumerate(iter(val_dl)):
        x, y = batch
        val_loss_value = val_loss(x, y, model, loss_fn)
        val_is_correct = accuracy(x, y, model)
    val_epoch_accuracy = np.mean(val_is_correct)
    
    # Lưu trữ kết quả sau mỗi Epoch
    train_losses.append(train_epoch_loss)
    train_accuracies.append(train_epoch_accuracy)
    val_losses.append(val_loss_value)
    val_accuracies.append(val_epoch_accuracy)

NameError: name 'get_data' is not defined

In [ ]:
# Cấu hình hiển thị đồ thị qua 10 Epochs
epochs = np.arange(10) + 1
plt.figure(figsize=(10, 10))

# Biểu đồ so sánh Loss cải thiện của SGD
plt.subplot(211)
plt.plot(epochs, train_losses, 'bo', label='Training loss')
plt.plot(epochs, val_losses, 'r', label='Validation loss')
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.title('Training and validation loss with SGD optimizer')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(False)

# Biểu đồ so sánh Accuracy đạt được của SGD
plt.subplot(212)
plt.plot(epochs, train_accuracies, 'bo', label='Training accuracy')
plt.plot(epochs, val_accuracies, 'r', label='Validation accuracy')
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.title('Training and validation accuracy with SGD optimizer')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x * 100) for x in plt.gca().get_yticks()])
plt.legend()
plt.grid(False)

plt.tight_layout()
plt.show()